# Sentiment / Emotion Analysis (Naive Bayes & SVM)

Dataset: nlp_dataset.csv
Columns used:
- Comment (text)
- Emotion (label)

This notebook covers preprocessing, TF-IDF feature extraction, model training, and evaluation.

---

In [3]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\akmsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:

# Load dataset
df = pd.read_csv("nlp_dataset.csv")

df.head(), df['Emotion'].value_counts()


(                                             Comment Emotion
 0  i seriously hate one subject to death but now ...    fear
 1                 im so full of life i feel appalled   anger
 2  i sit here to write i start to dig out my feel...    fear
 3  ive been really angry with r and i feel like a...     joy
 4  i feel suspicious if there is no one outside l...    fear,
 Emotion
 anger    2000
 joy      2000
 fear     1937
 Name: count, dtype: int64)

In [5]:

# Text preprocessing: lowercase, remove URLs/symbols, remove stopwords
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return " ".join(tokens)

df['clean_text'] = df['Comment'].apply(clean_text)
df[['Comment','clean_text']].head()


,Comment,clean_text
0,i seriously hate one subject to death but now ...,seriously hate one subject death feel reluctan...
1,im so full of life i feel appalled,im full life feel appalled
2,i sit here to write i start to dig out my feel...,sit write start dig feelings think afraid acce...
3,ive been really angry with r and i feel like a...,ive really angry r feel like idiot trusting fi...
4,i feel suspicious if there is no one outside l...,feel suspicious one outside like rapture happe...


In [6]:

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['Emotion'],
    test_size=0.2, random_state=42, stratify=df['Emotion']
)


In [7]:

# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

X_train_vec.shape, X_test_vec.shape


((4749, 5000), (1188, 5000))

In [8]:

# -------- Naive Bayes --------
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

nb_pred = nb_model.predict(X_test_vec)
nb_acc = accuracy_score(y_test, nb_pred)
nb_f1 = f1_score(y_test, nb_pred, average='weighted')

print("Naive Bayes Accuracy:", nb_acc)
print("Naive Bayes F1-score:", nb_f1)
print(classification_report(y_test, nb_pred))


Naive Bayes Accuracy: 0.8981481481481481
Naive Bayes F1-score: 0.8981749828366568
              precision    recall  f1-score   support

       anger       0.88      0.91      0.90       400
        fear       0.89      0.91      0.90       388
         joy       0.92      0.88      0.90       400

    accuracy                           0.90      1188
   macro avg       0.90      0.90      0.90      1188
weighted avg       0.90      0.90      0.90      1188



In [9]:

# -------- Support Vector Machine --------
svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)

svm_pred = svm_model.predict(X_test_vec)
svm_acc = accuracy_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred, average='weighted')

print("SVM Accuracy:", svm_acc)
print("SVM F1-score:", svm_f1)
print(classification_report(y_test, svm_pred))


SVM Accuracy: 0.9393939393939394
SVM F1-score: 0.9393654636933793
              precision    recall  f1-score   support

       anger       0.95      0.93      0.94       400
        fear       0.94      0.94      0.94       388
         joy       0.93      0.95      0.94       400

    accuracy                           0.94      1188
   macro avg       0.94      0.94      0.94      1188
weighted avg       0.94      0.94      0.94      1188



In [10]:

# Model comparison
results = pd.DataFrame({
    "Model": ["Naive Bayes", "SVM"],
    "Accuracy": [nb_acc, svm_acc],
    "F1-score": [nb_f1, svm_f1]
})

results


,Model,Accuracy,F1-score
0,Naive Bayes,0.898148,0.898175
1,SVM,0.939394,0.939365



## Explanation of Results

**Preprocessing:** Text was lowercased, special characters removed, and stopwords filtered. This reduces noise and improves learning by focusing on meaningful words.

**Feature Extraction:** TF-IDF converts text into numerical vectors based on word importance across documents, helping models distinguish emotions more effectively.

**Models:** Naive Bayes is fast and works well for text due to probabilistic assumptions. SVM usually provides stronger performance by finding optimal decision boundaries. The better model can be selected using Accuracy and F1-score shown above.
